# RCSB PDB structures for thrombin (UniProt entry name lookup)

`chem.rcsb.download_structures` accepts the same three id forms as
`chem.chembl.download_activities` (ChEMBL target id, UniProt accession, or
UniProt entry name). It resolves the id to a UniProt accession, finds every
PDB entry annotated with that accession via the RCSB Search API, optionally
filters by resolution (entries without one, e.g. NMR structures, are dropped
whenever a threshold is given), and downloads each entry's structure file(s).
Files already present in `outdir` are left alone, so re-running only fetches
what's missing.

In [1]:
from chem import rcsb

n = rcsb.download_structures(
    "THRB_HUMAN",
    resolution_thres=1.5,
    filetype="pdb",
    outdir="data",
)
n

download_structures(id='THRB_HUMAN', resolution_thres=1.5, outdir='data', filetype='pdb')
downloading structures to data: 100%|█████████████████████████████████████████████████████████████████| 66/66 [00:00<00:00, 38798.05entry/s]
wrote 66 structures to data


66

### View one of the downloaded structures

In [2]:
import os

import ipywidgets as widgets
from IPython.display import HTML, display

from chem import view3d
from chem.protein import SOLVENT_AND_IONS

# NAG (glycosylation) and TYS (sulfated hirudin-fragment tyrosine, a covalently
# peptide-bonded modified residue) aren't solvent, but they're also part of the
# ordinary modeled structure, not "the ligand" -- excluding them keeps them on
# the plain cartoon instead of popping out as isolated atom-level sticks.
_display_exclude = SOLVENT_AND_IONS | {"NAG", "TYS", "MRD"}

pdb_files = sorted(f for f in os.listdir("data") if f.endswith(".pdb"))
print(len(pdb_files), "structures downloaded")

# ".pdb" is common to every option, so show just the 4-character PDB id as the
# button label while keeping the full filename as the underlying value.
picker = widgets.ToggleButtons(
    options=[(os.path.splitext(f)[0], f) for f in pdb_files],
    value=pdb_files[0],
    description="PDB:",
    style={"button_width": "44px"},
    layout=widgets.Layout(width="100%", display="flex", flex_flow="row wrap"),
)
# Tighter padding/font than ipywidgets' default so more buttons fit per row --
# there can be dozens of structures. ToggleButtonsStyle has no padding/font_size
# trait, so this needs raw CSS scoped to a class on just this widget.
picker.add_class("chem-compact-toggle")
display(HTML(
    "<style>.chem-compact-toggle .widget-toggle-button "
    "{ padding: 1px 3px; font-size: 11px; min-width: 0; } </style>"
))
output = widgets.Output()


def render(pdb_filename):
    output.clear_output(wait=True)
    with output:
        view3d.render_protein(os.path.join("data", pdb_filename), exclude=_display_exclude)


picker.observe(lambda change: render(change["new"]) if change["name"] == "value" else None, names="value")
render(picker.value)

display(widgets.VBox([picker, output]))

66 structures downloaded


## Check by visual inspection for reasonable ligand bound (manual)

In [ ]:
1A3B 1A3E

## AlphaFold DB predicted structure for thrombin (UniProt entry name lookup)

`chem.alphafold.download_structures` accepts the same three id forms as
`chem.chembl.download_activities` and `chem.rcsb.download_structures`. It
resolves the id to a UniProt accession and downloads every AlphaFold DB
prediction entry for it (usually one, but very large proteins may be split
into fragments, and some targets without an official prediction have
community-submitted alternatives instead), optionally filtered by a minimum
average pLDDT confidence (`plddt_thres`, 0-100). As with `chem.rcsb`, files
already present in `outdir` are left alone.

In [ ]:
from chem import alphafold

n = alphafold.download_structures(
    "THRB_HUMAN",
    outdir="af_data",
    filetype="pdb",
)
n

### View the predicted structure, colored by pLDDT confidence

In [ ]:
import os

from chem import view3d

pdb_files = sorted(f for f in os.listdir("af_data") if f.endswith(".pdb"))
print(pdb_files)

# AlphaFold stores per-residue pLDDT confidence in the B-factor column.
view3d.render_protein(
    os.path.join("af_data", pdb_files[0]), coloring="bfactor", width=600, height=600
)

## Aligning multiple thrombin structures with chem.protein.align

`chem.protein.align` sequence-aligns and structurally superposes a set of
same-target structures (PDB/CIF, RCSB/AlphaFold freely mixed) onto a
reference, selecting each structure's primary polymer chain automatically
(the one with the most standard amino acid residues -- e.g. thrombin's
catalytic heavy chain rather than its short light chain). Here every
downloaded RCSB structure is aligned onto the AlphaFold model (fixed as the
reference, since it doesn't need to be a member of `structures`). It writes
one PDB file per input into `outdir`, so each can be overlaid in the same
viewer.

In [ ]:
import glob
import itertools
import os

import ipywidgets as widgets
import pandas as pd
import py3Dmol
from IPython.display import display

from chem import protein
from chem.protein import SOLVENT_AND_IONS

# NAG (glycosylation) and TYS (sulfated hirudin-fragment tyrosine, a covalently
# peptide-bonded modified residue) aren't solvent, but they're also part of the
# ordinary modeled structure, not "the ligand" -- excluding them keeps them on
# the plain cartoon instead of popping out as isolated atom-level sticks.
_display_exclude = SOLVENT_AND_IONS | {"NAG", "TYS", "MRD"}

# Reference is fixed to the AlphaFold model -- not user-selectable.
reference_path = glob.glob("af_data/*.pdb")[0]
reference_id = os.path.splitext(os.path.basename(reference_path))[0]

structures = sorted(glob.glob("data/*.pdb"))
structure_ids = [os.path.splitext(os.path.basename(p))[0] for p in structures]
id_to_path = dict(zip(structure_ids, structures))

# Align every downloaded RCSB structure onto the fixed AlphaFold reference, once.
# align() already returns plain (not numpy) floats rounded to 3dp.
rmsd = protein.align(structures, reference=reference_path, outdir="aligned")
rmsd_df = pd.DataFrame(
    [{"id": os.path.splitext(os.path.basename(p))[0], "rmsd": r} for p, r in rmsd.items()]
).sort_values("rmsd")
display(rmsd_df)

# One independently-toggleable checkbox-style button per structure (unlike
# ToggleButtons, several can be selected at once) for choosing which of the
# already-aligned structures to overlay in the viewer below.
_default_overlay = set(structure_ids[:4])
overlay_toggles = {
    i: widgets.ToggleButton(description=i, value=(i in _default_overlay), layout=widgets.Layout(width="65px"))
    for i in structure_ids
}
overlay_box = widgets.GridBox(
    list(overlay_toggles.values()),
    layout=widgets.Layout(grid_template_columns="repeat(auto-fill, 65px)", width="100%"),
)

output = widgets.Output()
# 3Dmol.js only defines "*Carbon" colorscheme presets for these 8 colors.
_COLORS = ["orange", "cyan", "magenta", "yellow", "green", "purple", "blue", "white"]


def render_overlay(*_):
    output.clear_output(wait=True)
    with output:
        chosen_ids = [i for i, t in overlay_toggles.items() if t.value]
        if not chosen_ids:
            print("Select at least one structure to overlay.")
            return

        view = py3Dmol.view(width=600, height=450)
        # itertools.cycle so the color palette never silently runs out and drops
        # a selected structure, however many are chosen. The (fixed) reference is
        # always shown first.
        for struct_id, color in zip([reference_id] + chosen_ids, itertools.cycle(_COLORS)):
            aligned_path = os.path.join("aligned", f"{struct_id}.pdb")
            with open(aligned_path) as f:
                pdb_text = f.read()
            view.addModel(pdb_text, "pdb")
            view.setStyle({"model": -1}, {"cartoon": {"color": color}})

            # Cartoon only draws the polymer backbone, so ligands need an explicit
            # style; skip water/ions/common crystallization additives.
            ligand_resnames = sorted(
                {line[17:20].strip() for line in pdb_text.splitlines() if line.startswith("HETATM")}
                - _display_exclude
            )
            if ligand_resnames:
                view.addStyle(
                    {"model": -1, "resn": ligand_resnames}, {"stick": {"colorscheme": f"{color}Carbon"}}
                )
        view.zoomTo()
        view.show()


for _toggle in overlay_toggles.values():
    _toggle.observe(render_overlay, names="value")
render_overlay()

display(widgets.VBox([overlay_box, output]))

### Choose which aligned structures to overlay

The reference is fixed to the AlphaFold model, and every downloaded RCSB
structure is aligned onto it up front (the RMSD table above). Toggle any
number of the resulting structures on to overlay them in the viewer --
independent `ToggleButton`s, so several can be selected at once, updating the
view immediately (no realignment needed, since `aligned/` already has every
structure).

## Finding the ligand pocket with chem.protein.find_pocket

`chem.protein.find_pocket` runs [fpocket](https://github.com/Discngine/fpocket)
on a PDB file and picks the pocket closest to a ligand, returning fpocket's
score/druggability/volume plus the residues lining it.

`4UEH.pdb`'s ligand `BEN` (benzamidine) was tried first, but it's a minimal
fragment that only probes the S1 specificity pocket -- and most of the
highest-resolution structures in this download turn out to be from a
fragment-screening campaign explicitly built around small S1-pocket-only
probes (see `2BVR`'s title), not full active-site-spanning inhibitors.
`3RM0.pdb`'s ligand `S54` (N-(benzylsulfonyl)-D-valyl-N-(4-carbamimidoylbenzyl)-
L-prolinamide -- a real, self-contained, LINK-free designed inhibitor
spanning the S1-S3 subsites) gives a noticeably more complete picture at
comparable resolution (1.34 vs 1.16 A): 27 residues vs 21, and a higher
druggability score (0.874 vs 0.767).

In [ ]:
pocket = protein.find_pocket("data/3RM0.pdb", ligand="S54")
print("pocket_id:", pocket["pocket_id"])
print("score:", pocket["score"], "druggability_score:", pocket["druggability_score"])
print("volume:", pocket["volume"])
pocket["residues"]

### Consensus active site residues across every aligned structure

Rather than trusting a single structure's pocket, run `find_pocket` on every
aligned RCSB structure that has a usable ligand, and count how often each
residue (by chymotrypsin-numbered `resnum`/`icode`, which is consistent
across thrombin PDB entries) shows up in the detected pocket. Per structure,
the ligand is auto-picked as its largest HETATM group excluding solvent/ions
(`chem.protein.SOLVENT_AND_IONS`) *and* the glycosylation sugar `NAG` /
sulfotyrosine `TYS` common throughout this hirudin-bound dataset, neither of
which is the pharmacological ligand. Structures with no such group (no bound
ligand) are skipped.

In [ ]:
import os

import pandas as pd
from tqdm.notebook import tqdm

from chem.protein import SOLVENT_AND_IONS

# NAG (glycosylation) and TYS (sulfated hirudin-fragment tyrosine) aren't solvent,
# but they aren't the pharmacological ligand either -- exclude them too when
# picking each structure's ligand.
_ligand_exclude = SOLVENT_AND_IONS | {"NAG", "TYS", "MRD"}


def _pick_ligand_code(path):
    """The resname of the largest non-excluded HETATM group in a PDB file, or None."""
    counts = {}
    with open(path) as f:
        for line in f:
            if line.startswith("HETATM"):
                resname = line[17:20].strip()
                if resname in _ligand_exclude:
                    continue
                counts[resname] = counts.get(resname, 0) + 1
    return max(counts, key=counts.get) if counts else None


consensus_counts = {}  # (resnum, icode) -> {"resname": ..., "count": int}
n_analyzed = 0
skipped = []

for struct_id in tqdm(structure_ids, desc="running find_pocket on aligned structures"):
    aligned_path = os.path.join("aligned", f"{struct_id}.pdb")
    ligand_code = _pick_ligand_code(aligned_path)
    if ligand_code is None:
        skipped.append((struct_id, "no non-solvent ligand"))
        continue
    try:
        pocket = protein.find_pocket(aligned_path, ligand=ligand_code)
    except Exception as e:
        skipped.append((struct_id, str(e)))
        continue

    n_analyzed += 1
    for r in pocket["residues"]:
        key = (r["resnum"], r["icode"])
        entry = consensus_counts.setdefault(key, {"resname": r["resname"], "count": 0})
        entry["count"] += 1

print(f"\nanalyzed {n_analyzed} structures, skipped {len(skipped)}:")
for struct_id, reason in skipped:
    print(f"  {struct_id}: {reason}")

### Residue frequency across structures

In [ ]:
consensus_df = pd.DataFrame(
    [
        {
            "resname": v["resname"],
            "resnum": resnum,
            "icode": icode,
            "count": v["count"],
            "fraction": round(v["count"] / n_analyzed, 3),
        }
        for (resnum, icode), v in consensus_counts.items()
    ]
).sort_values(["fraction", "resnum"], ascending=[False, True])
display(consensus_df)

### Consensus active site: adjustable threshold, structure viewer

`threshold` is the minimum fraction of analyzed structures a residue must
appear in (fpocket's detected pocket) to count as "consensus active site".
Lower it to widen the site (e.g. for MD or vina docking flexible-residue
selection); raise it to tighten to only the most consistently-detected
residues. `consensus_resnums`/`consensus_active_site` (recomputed live as
the slider moves) are the definitive list to export downstream. Pick which
aligned structure to view the result on, including the AlphaFold model --
its `resnum`s differ from the chymotrypsin-numbered RCSB convention the
consensus is computed in (sequential from the UniProt precursor's start,
1-622), so a resnum->resnum lookup is built once via the same
sequence-alignment machinery `chem.protein.align` uses internally,
referenced against `3RM0` (the structure the consensus numbers come from).

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, display

from chem.protein.structural_align import _chain_seq_and_ca, _load_structure, _matched_ca_pairs, _select_chain

# NAG (glycosylation) and TYS (sulfated hirudin-fragment tyrosine, a covalently
# peptide-bonded modified residue) aren't solvent, but they're also part of the
# ordinary modeled structure, not "the ligand" -- excluding them keeps them on
# the plain cartoon instead of popping out as isolated atom-level sticks.
_display_exclude = SOLVENT_AND_IONS | {"NAG", "TYS", "MRD"}

# resnum -> resnum lookup from the chymotrypsin-numbered reference (3RM0) to the
# AlphaFold model's own UniProt-sequential numbering, via sequence alignment.
_chymo_structure = _load_structure("aligned/3RM0.pdb")
_chymo_chain = _select_chain(next(_chymo_structure.get_models()))
_chymo_seq, _chymo_ca = _chain_seq_and_ca(_chymo_chain)

_af_structure = _load_structure(os.path.join("aligned", f"{reference_id}.pdb"))
_af_chain = _select_chain(next(_af_structure.get_models()))
_af_seq, _af_ca = _chain_seq_and_ca(_af_chain)

_chymo_pts, _af_pts = _matched_ca_pairs(_chymo_seq, _chymo_ca, _af_seq, _af_ca)
_chymo_to_af_resnum = {
    (a.get_parent().id[1], a.get_parent().id[2].strip()): b.get_parent().id[1]
    for a, b in zip(_chymo_pts, _af_pts)
}

threshold_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=1.0,
    step=0.05,
    description="Threshold:",
    readout_format=".2f",
    continuous_update=False,
)
structure_picker = widgets.ToggleButtons(
    options=[(f"{reference_id} (AlphaFold)", reference_id)] + [(i, i) for i in structure_ids],
    value="3RM0",
    description="Structure:",
    style={"button_width": "80px"},
    layout=widgets.Layout(width="100%", display="flex", flex_flow="row wrap"),
)
# Tighter padding/font than ipywidgets' default so more buttons fit per row.
structure_picker.add_class("chem-compact-toggle")
display(HTML(
    "<style>.chem-compact-toggle .widget-toggle-button "
    "{ padding: 1px 3px; font-size: 11px; min-width: 0; } </style>"
))
consensus_output = widgets.Output()

consensus_active_site = None
consensus_resnums = None


def render_consensus(*_):
    global consensus_active_site, consensus_resnums
    consensus_output.clear_output(wait=True)
    with consensus_output:
        consensus_active_site = consensus_df[consensus_df["fraction"] >= threshold_slider.value].sort_values(
            ["resnum", "icode"]
        )
        consensus_resnums = [(r.resnum, r.icode) for r in consensus_active_site.itertuples()]
        labels = [f"{r.resname.capitalize()}{r.resnum}{r.icode}" for r in consensus_active_site.itertuples()]
        print(
            f"{len(labels)} consensus active site residues "
            f"(present in >={threshold_slider.value * 100:.0f}% of {n_analyzed} analyzed structures):"
        )
        print(", ".join(labels))

        struct_id = structure_picker.value
        if struct_id == reference_id:
            resnums = sorted(
                {_chymo_to_af_resnum[key] for key in consensus_resnums if key in _chymo_to_af_resnum}
            )
        else:
            resnums = sorted({resnum for resnum, _icode in consensus_resnums})

        path = os.path.join("aligned", f"{struct_id}.pdb")
        with open(path) as f:
            pdb_text = f.read()

        ligand_resnames = sorted(
            {line[17:20].strip() for line in pdb_text.splitlines() if line.startswith("HETATM")}
            - _display_exclude
        )

        view = py3Dmol.view(width=600, height=450)
        view.addModel(pdb_text, "pdb")
        # Translucent backbone so the highlighted residues stand out.
        view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.5}})
        if resnums:
            view.addStyle({"resi": resnums}, {"stick": {"colorscheme": "greenCarbon"}})
            view.addResLabels(
                {"resi": resnums},
                {
                    "font": "sans-serif",
                    "fontSize": 12,
                    "fontColor": "black",
                    "showBackground": True,
                    "backgroundColor": "white",
                    "backgroundOpacity": 0.7,
                },
            )
        if ligand_resnames:
            view.addStyle({"resn": ligand_resnames}, {"stick": {"colorscheme": "orangeCarbon"}})
        view.zoomTo({"resi": resnums} if resnums else {})
        view.show()


threshold_slider.observe(render_consensus, names="value")
structure_picker.observe(render_consensus, names="value")
render_consensus()

display(widgets.VBox([threshold_slider, structure_picker, consensus_output]))